In [13]:

import wfdb
import numpy as np 
import os 
from scipy.fft import fft, fftfreq
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.io import loadmat
from scipy.signal import windows, periodogram,welch, spectrogram
import numpy as np
import mne

os.makedirs(os.path.join(os.getcwd(),'images'),exist_ok=True)
os.makedirs(os.path.join(os.getcwd(),'datasets'),exist_ok=True)
IMAGE_DIR = os.path.join(os.getcwd(),'images')
DATASETS = os.path.join(os.getcwd(),'datasets')

record = wfdb.rdrecord(os.path.join(DATASETS,'ath_001'))  # reads 100.dat + 100.hea

signals = record.p_signal.T          # shape: (n_channels, n_samples)
sfreq = record.fs
ch_names = record.sig_name

info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types=['ecg']*len(ch_names))
raw = mne.io.RawArray(signals, info)
ecg_events, _, _ = mne.preprocessing.find_ecg_events(
    raw, ch_name='II'  # or None to auto-detect
)

# ecg_events[:, 0] are sample indices of R-peaks
r_peak_samples = ecg_events[:, 0]
r_peak_times = r_peak_samples / sfreq  

Creating RawArray with float64 data, n_channels=12, n_times=5000
    Range : 0 ... 4999 =      0.000 ...     9.998 secs
Ready.
Using channel II to identify heart beats.
Setting up band-pass filter from 5 - 35 Hz

FIR filter parameters
---------------------
Designing a two-pass forward and reverse, zero-phase, non-causal bandpass filter:
- Windowed frequency-domain design (firwin2) method
- Hann window
- Lower passband edge: 5.00
- Lower transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 4.75 Hz)
- Upper passband edge: 35.00 Hz
- Upper transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 35.25 Hz)
- Filter length: 5000 samples (10.000 s)

Number of ECG events detected : 9 (average pulse 54.0 / min.)


/tmp/ipykernel_27676/3201854080.py:25: RuntimeWarning:

filter_length (8192) is longer than the signal (5000), distortion is likely. Reduce filter length or filter a longer signal.



In [39]:
for comment in record.comments:
    c = comment.split(":")
    source = c[0]
    text = c[-1]
    print (f'\n{source} reported: {text}')


SL12 reported:  Sinus bradycardia with marked sinus arrhythmia, Right axis deviation, Borderline ECG

C reported:  Sinus arrhythmia,  Normal ECG


In [ ]:
ecg_signal = raw.get_data(picks=['II'])[0]
time = np.arange(1, len(ecg_signal) ) * (1 / sfreq) 
fig_raw = go.Figure()
fig_raw.add_trace(go.Scatter(
    x=time,
    y=ecg_signal,
    mode="lines",
    name=f"ECG {lead_name}"
))
fig_raw.update_layout(
    title=f"Raw ECG signal (lead {lead_name})",
    xaxis_title="Time (s)",
    yaxis_title="Amplitude"
)
fig_raw.show()

array([-0.02192, -0.05116, -0.07552, ...,  0.00242,  0.00242,  0.00242],
      shape=(5000,))

In [ ]:
import numpy as np
from scipy.signal import welch
import plotly.graph_objects as go

# choose ECG lead (you already used 'II')
lead_name = 'II'
lead_idx = ch_names.index(lead_name)
ecg = signals[lead_idx]
n_samples = ecg.shape[0]
times = np.arange(n_samples) / sfreq



# -----------------------------
# 1) Plot ECG segment + R-peaks
# -----------------------------
seg_dur = 10.0         # seconds to show
seg_start = 0
seg_stop = int(seg_dur * sfreq)

t_seg = times[seg_start:seg_stop]
ecg_seg = ecg[seg_start:seg_stop]

# R-peaks that fall into the chosen segment
mask = (r_peak_samples >= seg_start) & (r_peak_samples < seg_stop)
r_seg = r_peak_samples[mask]
t_peaks = r_seg / sfreq
ecg_peaks = ecg[r_seg]

fig_ecg = go.Figure()
fig_ecg.add_trace(go.Scatter(
    x=t_seg, y=ecg_seg, mode="lines", name="ECG"
))
fig_ecg.add_trace(go.Scatter(
    x=t_peaks, y=ecg_peaks,
    mode="markers", name="R-peaks",
    marker=dict(color="red", size=8)
))
fig_ecg.update_layout(
    title=f"ECG lead {lead_name} with R-peaks",
    xaxis_title="Time (s)",
    yaxis_title="Amplitude"
)
fig_ecg.show()

# -----------------------------
# 2) Extract and plot heartbeats
# -----------------------------
# window around each R-peak (e.g. 300 ms before, 400 ms after)
win_before = int(0.3 * sfreq)
win_after = int(0.4 * sfreq)
beat_len = win_before + win_after

beats = []
for s in r_peak_samples:
    start = s - win_before
    stop = s + win_after
    if start < 0 or stop > n_samples:
        continue
    beats.append(ecg[start:stop])

beats = np.array(beats)  # shape: (n_beats, beat_len)
beat_t = np.arange(-win_before, win_after) / sfreq

fig_beats = go.Figure()
n_plot = min(20, beats.shape[0])   # overlay first 20 beats
for i in range(n_plot):
    fig_beats.add_trace(go.Scatter(
        x=beat_t,
        y=beats[i],
        mode="lines",
        line=dict(width=1),
        opacity=0.5,
        showlegend=False
    ))
fig_beats.update_layout(
    title=f"Overlaid heartbeats (lead {lead_name})",
    xaxis_title="Time from R-peak (s)",
    yaxis_title="Amplitude"
)
fig_beats.show()

# -----------------------------
# 3) PSD (Welch) and Plotly plot
# -----------------------------

WINDOW = 4 *int(sfreq)         
NOVERLAP = WINDOW//2       
NFFT = WINDOW          
window = windows.hamming(WINDOW)  
f, Pxx = welch(
    ecg,
    fs=sfreq,
    nperseg=int(4 * sfreq),
    window=window,
    nperseg=WINDOW,
    noverlap=NOVERLAP,
    nfft=NFFT,
    scaling='density',# 4‑s windows is typical for ECG PSD [web:14]
)

fig_psd = go.Figure()
fig_psd.add_trace(go.Scatter(
    x=f, y=Pxx, mode="lines", name="PSD"
))
fig_psd.update_layout(
    title=f"ECG Power Spectral Density (Welch, {lead_name})",
    xaxis_title="Frequency (Hz)",
    yaxis_title="PSD (power/Hz)",
    yaxis_type="log"
)
fig_psd.show()


In [29]:
WINDOW = 4 *int(sfreq)         
NOVERLAP = WINDOW//2       
NFFT = WINDOW          
window = windows.hamming(WINDOW)  
f, Pxx = welch(
    ecg,
    fs=sfreq,
    window=window,
    nperseg=WINDOW,
    noverlap=NOVERLAP,
    nfft=NFFT,
    scaling='density',# 4‑s windows is typical for ECG PSD [web:14]
)

fig_psd = go.Figure()
fig_psd.add_trace(go.Scatter(
    x=f, y=Pxx, mode="lines", name="PSD"
))
fig_psd.update_layout(
    title=f"ECG Power Spectral Density (Welch, {lead_name})",
    xaxis_title="Frequency (Hz)",
    yaxis_title="PSD (power/Hz)",
    
)
fig_psd.show()


In [16]:
# Remove DC component (mean)
ecg_signal = ecg_signal - np.mean(ecg_signal)
N = len(ecg_signal)
X = fft(ecg_signal)
two_sided = np.abs(X) / N
one_sided = two_sided[:N//2 + 1]
one_sided[1:-1] *= 2             # double non‑DC, non‑Nyquist bins

freqs = sfreq * np.arange(0, N//2 + 1) / N
f = sfreq * np.arange(0, len(ecg_signal)//2 + 1) / len(ecg_signal)


fig = go.Figure()

fig.add_trace(go.Scatter(
    x=f,
    y=one_sided,
    mode='lines',
    name='EEG trace'
))

fig.update_layout(
    title='Single-Sided Amplitude Spectrum',
    xaxis_title='f (Hz)',
    yaxis_title='Magnitude (μV)',
    #xaxis=dict(range=[0, 50]),  # Equivalent to xlim([0 16])
    #yaxis=dict(range=[0, 10]),  # Equivalent to xlim([0 16])
    template='plotly_white'
)
fig.show()


In [17]:
faxis, pxx = periodogram(
    ecg_signal,
    fs=sfreq,
    window=windows.hamming(len(ecg_signal)),
    nfft=len(ecg_signal),
    scaling='density',
    return_onesided=True
)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=faxis,
    y=pxx,
    mode='lines',
    name='EEG trace'
))

fig.update_layout(
    title='periodogram',
    xaxis_title='f (Hz)',
    yaxis_title='|P(db)|',
    xaxis=dict(range=[0, 50]),  # Equivalent to xlim([0 16])
    template='plotly_white',
    
)
fig.show()

In [40]:
sfreq

500

In [42]:

WINDOW = 4 *int(sfreq)         
NOVERLAP = WINDOW//2       
NFFT = WINDOW          
window = windows.hamming(WINDOW)  

faxis, pxx = welch(
    ecg_signal,
    fs=sfreq,
    window=window,
    nperseg=WINDOW,
    noverlap=NOVERLAP,
    nfft=NFFT,
    scaling='density',
    return_onesided=True
)
#pxx_db = 10 * np.log10(pxx)
fig = go.Figure()
fig.add_trace(go.Scatter(x=faxis, y=pxx, mode='lines'))
fig.update_layout(
    title=f'pwelch {NOVERLAP} samples overlap',
    yaxis_title='PSD (V^2/Hz)',
    xaxis_title='Frequency (Hz)',
    template='plotly_white',
    
)
fig.update_xaxes(range=[0, 40])
#fig.write_image(os.path.join(IMAGE_DIR,f"{signal_name}_pwelch_{NOVERLAP}_samples.png"), width=800, height=500, scale=2)
fig.show()

In [26]:
f, t, Sxx = spectrogram(ecg_signal, fs=sfreq, nperseg=WINDOW//2, noverlap=NOVERLAP//2)
#normalized_freq = f / np.max(f)
fig = go.Figure()

fig.add_trace(
    go.Heatmap(
        x=t,              # use corrected time base
        y=f,
        z=Sxx,
        colorscale='Viridis',
        colorbar=dict(title='Power (dB)', orientation='h'),
        name='Spectrogram'
    ),
)

fig.update_layout(
    height=800,
    width=1500,
    showlegend=True,
    xaxis_title='Time(s)',
    yaxis_title='Frequency(Hz)',
)
#fig.write_image(os.path.join(IMAGE_DIR,f"{signal_name}_spectogram.png"), width=600, height=500, scale=2)
fig.show()
